# Master Pipeline: LLIE + YOLOv11n on ExDark

**Evaluasi Metode Low-Light Image Enhancement sebagai Preprocessing untuk Object Detection pada Dataset ExDark**

| Scenario | Enhancement | Training Data | Test Data |
|----------|-------------|---------------|----------|
| S1_Raw | None (Baseline) | Raw | Raw |
| S2_HVI_CIDNet | HVI-CIDNet | Enhanced | Enhanced |
| S3_RetinexFormer | RetinexFormer | Enhanced | Enhanced |
| S4_LYT_Net | LYT-Net | Enhanced | Enhanced |

---
**How to use (S1-first approach):**
1. Set `QUICK_TEST = True` for a quick 1-epoch validation run
2. Set `ACTIVE_SCENARIOS` to **only S1** (default) — run all cells top-to-bottom
3. Once S1 is validated, edit `ACTIVE_SCENARIOS` to add S2/S3/S4
4. For S2-S4: also run the Fase 2 enhancement cells (2a, 2b, 2c) and Fase 3 training cells (3b, 3c, 3d)
5. Finally set `QUICK_TEST = False` for the full 100-epoch experiment

**Cell run order for S1 only:**
> Cell 0.1 → 0.2 → 0.3 → Fase 1 (all) → *skip Fase 2* → Fase 3a only → Fase 4+ (all, loops auto-filter to active scenarios)

## 0. Configuration & Environment Setup

In [ ]:
#@title 0.1 Mount Google Drive & Clone Repo
from google.colab import drive
drive.mount('/content/drive')

# === CONFIGURATION ===
QUICK_TEST = True  # @param {type:"boolean"}
REPO_URL = "https://github.com/Otachiking/Object-Detection-ExDARK-with-LLIE.git"  # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/KULIAH-S1INFORMATIKA/TA-IQBAL"  # @param {type:"string"}

# === SCENARIO SELECTOR ===
# For S1-first approach: keep only S1 active, uncomment others later
ACTIVE_SCENARIOS = [
    ("s1_raw", "S1_Raw"),
    # ("s2_hvi_cidnet", "S2_HVI_CIDNet"),       # ← Uncomment after S1 validated
    # ("s3_retinexformer", "S3_RetinexFormer"),  # ← Uncomment after S1 validated
    # ("s4_lyt_net", "S4_LYT_Net"),              # ← Uncomment after S1 validated
]

# All 4 scenarios (for reference)
ALL_SCENARIOS = [
    ("s1_raw", "S1_Raw"),
    ("s2_hvi_cidnet", "S2_HVI_CIDNet"),
    ("s3_retinexformer", "S3_RetinexFormer"),
    ("s4_lyt_net", "S4_LYT_Net"),
]

# Clone or pull repo
import os
import subprocess

REPO_DIR = "/content/TA-IQBAL-ObjectDetectionExDARKwithLLIE"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Repo exists, resetting to latest origin/main...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
else:
    if os.path.exists(REPO_DIR):
        import shutil
        shutil.rmtree(REPO_DIR)
    print(f"Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print(f"\nWorking directory: {os.getcwd()}")
print(f"Drive root: {DRIVE_ROOT}")
print(f"Quick test: {QUICK_TEST}")
print(f"Active scenarios: {[s[1] for s in ACTIVE_SCENARIOS]}")

# Sanity check: verify src exists
assert os.path.isfile(os.path.join(REPO_DIR, "src", "data", "split_dataset.py")), \
    "src/data/split_dataset.py missing after clone! Check that 'src/' is pushed to GitHub."
print("✓ Source files verified")


In [ ]:
#@title 0.2 Install Dependencies
!pip install -q ultralytics pyiqa thop fvcore scipy pandas pyyaml seaborn tqdm gdown huggingface_hub

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"VRAM: {vram / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
#@title 0.3 Load Configuration
import os
import sys
sys.path.insert(0, REPO_DIR)

from src.config import load_config, save_config_snapshot, save_environment_info
from src.seed import set_global_seed

# Load S1 config (base) to get paths
cfg_s1 = load_config("s1_raw", quick_test=QUICK_TEST)
set_global_seed(cfg_s1["seed"])

# ---- Compatibility layer (old notebook keys vs new paths.yaml schema) ----
paths = cfg_s1.get("paths", {})
data_paths = paths.get("data", {})

# Main roots
OUTPUT_ROOT = paths.get("output_root") or paths.get("drive_root") or paths.get("project_root")
EXDARK_ROOT = paths.get("exdark_root") or data_paths.get("exdark_original")

if OUTPUT_ROOT is None:
    raise KeyError("Could not resolve OUTPUT_ROOT from config['paths']")
if EXDARK_ROOT is None:
    raise KeyError("Could not resolve EXDARK_ROOT from config['paths']['data']['exdark_original']")

# Backfill expected keys for downstream cells
cfg_s1.setdefault("paths", {})
cfg_s1["paths"]["output_root"] = OUTPUT_ROOT
cfg_s1["paths"]["exdark_root"] = EXDARK_ROOT

# Backfill old exdark_structure mapping if needed
if "exdark_structure" not in cfg_s1["paths"]:
    exdark_meta = cfg_s1.get("paths_meta", {}).get("exdark", {})
    cfg_s1["paths"]["exdark_structure"] = {
        "images": exdark_meta.get("images_dir", "Dataset/ExDark"),
        "groundtruth": exdark_meta.get("groundtruth_dir", "Groundtruth"),
        "classlist": exdark_meta.get("classlist_file", "Groundtruth/imageclasslist.txt"),
    }

print(f"Output root: {OUTPUT_ROOT}")
print(f"ExDark root: {EXDARK_ROOT}")
print(f"Seed: {cfg_s1['seed']}")

# Verify ExDark exists
assert os.path.exists(EXDARK_ROOT), f"ExDark not found at {EXDARK_ROOT}. Upload dataset to Drive first!"
print("\n✓ ExDark dataset found")

# Save environment info
os.makedirs(OUTPUT_ROOT, exist_ok=True)
save_environment_info(OUTPUT_ROOT)

---
## Fase 1: Data Preparation
Parse official split → Convert annotations → Build YOLO structure → Validate

In [ ]:
#@title Fase 1.1: Parse Official Split
import os
import sys

# Ensure repo is importable (REPO_DIR set by Cell 0.1)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.data.split_dataset import parse_split_file

classlist_path = os.path.join(
    EXDARK_ROOT,
    cfg_s1["paths"]["exdark_structure"]["groundtruth"],
    "imageclasslist.txt"
)
split_output = os.path.join(OUTPUT_ROOT, "splits")

splits = parse_split_file(classlist_path, split_output)
print(f"Repo: {REPO_DIR}")
print(f"Train: {splits['train']} | Val: {splits['val']} | Test: {splits['test']}")
print(f"Total: {splits['train'] + splits['val'] + splits['test']}")


In [ ]:
#@title Fase 1.2: Convert Annotations to YOLO Format
from src.data.convert_exdark import convert_exdark_to_yolo

gt_dir = os.path.join(EXDARK_ROOT, cfg_s1["paths"]["exdark_structure"]["groundtruth"])
img_dir = os.path.join(EXDARK_ROOT, cfg_s1["paths"]["exdark_structure"]["images"])
yolo_labels_dir = os.path.join(OUTPUT_ROOT, "ExDark_yolo_labels")

stats = convert_exdark_to_yolo(
    exdark_images_dir=img_dir,
    exdark_gt_dir=gt_dir,
    output_labels_dir=yolo_labels_dir,
)
print(f"\nImages: {stats['total_images']} | Labels: {stats['total_labels']} | "
      f"Objects: {stats['total_objects']} | Failed: {stats['failed']}")


In [ ]:
#@title Fase 1.3: Build YOLO Dataset Structure
from src.data.build_yolo_dataset import build_yolo_dataset

yolo_dir = os.path.join(OUTPUT_ROOT, "ExDark_yolo")

build_stats = build_yolo_dataset(
    exdark_images_dir=img_dir,
    labels_dir=yolo_labels_dir,
    splits_dir=split_output,
    output_dir=yolo_dir,
    target_size=cfg_s1["yolo"]["imgsz"],
)

total_processed = sum(s["processed"] for s in build_stats["splits"].values())
total_skipped = sum(s["skipped"] for s in build_stats["splits"].values())
print(f"\nProcessed: {total_processed} | Skipped: {total_skipped} | Errors: {len(build_stats['errors'])}")
print(f"Dataset YAML: {os.path.join(yolo_dir, 'dataset.yaml')}")


In [ ]:
#@title Fase 1.4: Validate Dataset
from src.data.validate_dataset import validate_yolo_dataset

val_results = validate_yolo_dataset(yolo_dir)

if val_results["valid"]:
    print("\n✓ Dataset validation PASSED")
    summary = val_results.get("summary", {})
    print(f"  Total images:  {summary.get('total_images', 0)}")
    print(f"  Total labels:  {summary.get('total_labels', 0)}")
    print(f"  Total objects: {summary.get('total_objects', 0)}")
else:
    print("\n✗ Validation FAILED:")
    for split_name, sd in val_results.get("splits", {}).items():
        if sd.get("error"):
            print(f"  - {split_name}: {sd['error']}")
        if sd.get("invalid_bbox", 0) > 0:
            print(f"  - {split_name}: {sd['invalid_bbox']} invalid bboxes")
        if sd.get("missing_labels", 0) > 0:
            print(f"  - {split_name}: {sd['missing_labels']} images without labels")


---
## Fase 2: Image Enhancement
Apply LLIE methods to generate enhanced datasets for S2, S3, S4.

**Note:** S1 (Raw) skips this phase entirely.

In [ ]:
#@title Fase 2: Enhancement Helper
from src.enhancement.run_enhancement import enhance_dataset, get_enhancer

def run_enhancement(scenario_config_name, enhancer_name):
    """Run enhancement for a single scenario."""
    cfg = load_config(scenario_config_name, quick_test=QUICK_TEST)
    set_global_seed(cfg["seed"])

    enhanced_dir = os.path.join(OUTPUT_ROOT, f"ExDark_enhanced_{enhancer_name}")
    cache_dir = os.path.join(OUTPUT_ROOT, "model_cache")

    print(f"\n{'='*50}")
    print(f"Enhancing with: {enhancer_name}")
    print(f"Output: {enhanced_dir}")
    print(f"{'='*50}")

    # Create and load enhancer
    enhancer = get_enhancer(enhancer_name, cache_dir)
    enhancer.load_model()

    stats = enhance_dataset(
        enhancer=enhancer,
        source_dataset_dir=yolo_dir,
        output_dir=enhanced_dir,
        yolo_labels_dir=yolo_dir,
    )

    print(f"\nDone: {stats['total_processed']} processed, "
          f"{stats['total_skipped']} skipped, {stats['total_failed']} failed")
    return stats


In [ ]:
#@title Fase 2a: S2 - HVI-CIDNet Enhancement
stats_s2 = run_enhancement("s2_hvi_cidnet", "hvi_cidnet")

In [ ]:
#@title Fase 2b: S3 - RetinexFormer Enhancement
stats_s3 = run_enhancement("s3_retinexformer", "retinexformer")

In [ ]:
#@title Fase 2c: S4 - LYT-Net Enhancement
stats_s4 = run_enhancement("s4_lyt_net", "lyt_net")

---
## Fase 3: YOLOv11n Training
Train one model per scenario. Each uses the same hyperparameters (only data differs).

**Locked params:** epochs=100, batch=16, imgsz=640, patience=20, seed=42

In [ ]:
#@title Fase 3: Training Helper
from src.training.train_yolo import train_yolo, get_best_weights

def run_training(scenario_config_name, scenario_name):
    """Train YOLO for a single scenario."""
    cfg = load_config(scenario_config_name, quick_test=QUICK_TEST)
    set_global_seed(cfg["seed"])

    # Determine dataset YAML
    enhancer_name = cfg.get("enhancer", {}).get("name", None)
    if enhancer_name and enhancer_name.lower() != "none":
        data_yaml = os.path.join(OUTPUT_ROOT, f"ExDark_enhanced_{enhancer_name}", "dataset.yaml")
    else:
        data_yaml = os.path.join(OUTPUT_ROOT, "ExDark_yolo", "dataset.yaml")

    assert os.path.exists(data_yaml), f"Dataset YAML not found: {data_yaml}"

    runs_dir = os.path.join(OUTPUT_ROOT, "runs")

    print(f"\n{'='*50}")
    print(f"Training: {scenario_name}")
    print(f"Data: {data_yaml}")
    print(f"Run dir: {runs_dir}/{scenario_name}")
    print(f"{'='*50}")

    results = train_yolo(
        dataset_yaml=data_yaml,
        scenario_name=scenario_name,
        output_dir=runs_dir,
        config=cfg,
    )

    # train_yolo already saves config snapshot and env info internally
    best = get_best_weights(os.path.join(runs_dir, scenario_name))
    print(f"Best weights: {best}")
    return results


In [ ]:
#@title Fase 3a: Train S1_Raw (Baseline)
results_s1 = run_training("s1_raw", "S1_Raw")

In [ ]:
#@title Fase 3b: Train S2_HVI_CIDNet
results_s2 = run_training("s2_hvi_cidnet", "S2_HVI_CIDNet")

In [ ]:
#@title Fase 3c: Train S3_RetinexFormer
results_s3 = run_training("s3_retinexformer", "S3_RetinexFormer")

In [ ]:
#@title Fase 3d: Train S4_LYT_Net
results_s4 = run_training("s4_lyt_net", "S4_LYT_Net")

---
## Fase 4: Detection Evaluation
Evaluate each trained model on the test split.

**Metrics:** mAP@0.5, mAP@0.5:0.95, Precision, Recall (overall + per-class)

In [ ]:
#@title Fase 4: Evaluation Helper
from src.evaluation.eval_yolo import evaluate_yolo

def run_evaluation(scenario_config_name, scenario_name):
    """Evaluate YOLO for a single scenario. Returns overall metrics dict."""
    cfg = load_config(scenario_config_name, quick_test=QUICK_TEST)
    set_global_seed(cfg["seed"])

    run_dir = os.path.join(OUTPUT_ROOT, "runs", scenario_name)
    weights_path = get_best_weights(run_dir)
    assert os.path.exists(weights_path), f"Weights not found: {weights_path}"

    enhancer_name = cfg.get("enhancer", {}).get("name", None)
    if enhancer_name and enhancer_name.lower() != "none":
        data_yaml = os.path.join(OUTPUT_ROOT, f"ExDark_enhanced_{enhancer_name}", "dataset.yaml")
    else:
        data_yaml = os.path.join(OUTPUT_ROOT, "ExDark_yolo", "dataset.yaml")

    eval_dir = os.path.join(OUTPUT_ROOT, "evaluation", scenario_name)

    results = evaluate_yolo(
        weights_path=weights_path,
        dataset_yaml=data_yaml,
        output_dir=eval_dir,
        scenario_name=scenario_name,
    )

    overall = results.get("overall", {})
    print(f"  mAP@0.5:      {overall.get('mAP_50', 0):.4f}")
    print(f"  mAP@0.5:0.95: {overall.get('mAP_50_95', 0):.4f}")
    print(f"  Precision:     {overall.get('precision', 0):.4f}")
    print(f"  Recall:        {overall.get('recall', 0):.4f}")

    # Return flattened overall dict for downstream (correlation, visualization)
    return overall


In [ ]:
#@title Fase 4: Run Evaluations (active scenarios only)
eval_results = {}
for sc, sn in ACTIVE_SCENARIOS:
    eval_results[sn] = run_evaluation(sc, sn)

# Summary table
import pandas as pd
summary_rows = []
for sn, res in eval_results.items():
    summary_rows.append({
        "Scenario": sn,
        "mAP@0.5": res.get("mAP_50", 0),
        "mAP@0.5:0.95": res.get("mAP_50_95", 0),
        "Precision": res.get("precision", 0),
        "Recall": res.get("recall", 0),
    })
pd.DataFrame(summary_rows)

---
## Fase 5: No-Reference Image Quality Assessment
Compute NIQE, BRISQUE, and LOE on enhanced test images.

**Note:** LOE requires raw-enhanced pairs, so it's only computed for S2-S4.

In [ ]:
#@title Fase 5: NR-IQA Metrics (active scenarios only)
from src.evaluation.nr_metrics import compute_nr_metrics

nr_results = {}
raw_test_dir = os.path.join(OUTPUT_ROOT, "ExDark_yolo", "images", "test")

for sc, sn in ACTIVE_SCENARIOS:
    cfg = load_config(sc, quick_test=QUICK_TEST)
    enhancer_name = cfg.get("enhancer", {}).get("name", None)
    has_enhancer = enhancer_name and enhancer_name.lower() != "none"

    if has_enhancer:
        test_dir = os.path.join(OUTPUT_ROOT, f"ExDark_enhanced_{enhancer_name}", "images", "test")
        raw_dir = raw_test_dir
    else:
        test_dir = raw_test_dir
        raw_dir = None

    nr_dir = os.path.join(OUTPUT_ROOT, "evaluation", sn, "nr_metrics")

    print(f"\n--- {sn} ---")
    nr_results[sn] = compute_nr_metrics(
        images_dir=test_dir,
        output_dir=nr_dir,
        scenario_name=sn,
        raw_images_dir=raw_dir,
    )

# Summary
nr_rows = []
for sn, res in nr_results.items():
    nr_rows.append({
        "Scenario": sn,
        "NIQE ↓": res.get("niqe_mean", None),
        "BRISQUE ↓": res.get("brisque_mean", None),
        "LOE ↓": res.get("loe_mean", None),
    })
pd.DataFrame(nr_rows)


---
## Fase 6: Latency & FLOPs Measurement
Measure end-to-end inference speed and computational cost.

In [ ]:
#@title Fase 6a: Inference Latency (active scenarios only)
from src.evaluation.latency import measure_latency

latency_results = {}

for sc, sn in ACTIVE_SCENARIOS:
    cfg = load_config(sc, quick_test=QUICK_TEST)
    enhancer_name = cfg.get("enhancer", {}).get("name", None)
    has_enhancer = enhancer_name and enhancer_name.lower() != "none"

    run_dir = os.path.join(OUTPUT_ROOT, "runs", sn)
    weights_path = get_best_weights(run_dir)

    lat_dir = os.path.join(OUTPUT_ROOT, "evaluation", sn, "latency")

    # Load enhancer object if needed
    enhancer_obj = None
    if has_enhancer:
        cache_dir = os.path.join(OUTPUT_ROOT, "model_cache")
        enhancer_obj = get_enhancer(enhancer_name, cache_dir)
        enhancer_obj.load_model()

    print(f"\n--- {sn} ---")
    raw = measure_latency(
        yolo_weights=weights_path,
        output_dir=lat_dir,
        scenario_name=sn,
        test_images_dir=raw_test_dir,
        enhancer=enhancer_obj,
        num_images=cfg.get("latency", {}).get("iterations", 200),
        warmup=cfg.get("latency", {}).get("warmup", 50),
    )

    # Normalize keys for downstream (visualization expects T_enhance_mean etc.)
    latency_results[sn] = {
        "T_enhance_mean": raw.get("T_enhance_ms_mean", 0),
        "T_detect_mean": raw.get("T_detect_ms_mean", 0),
        "T_total_mean": raw.get("T_total_ms_mean", 0),
    }

# Summary
lat_rows = []
for sn, res in latency_results.items():
    lat_rows.append({
        "Scenario": sn,
        "T_enhance (ms)": f"{res.get('T_enhance_mean', 0):.2f}",
        "T_detect (ms)": f"{res.get('T_detect_mean', 0):.2f}",
        "T_total (ms)": f"{res.get('T_total_mean', 0):.2f}",
    })
pd.DataFrame(lat_rows)


In [ ]:
#@title Fase 6b: FLOPs Computation (active scenarios only)
from src.evaluation.flops import compute_all_flops

flops_results = {}

for sc, sn in ACTIVE_SCENARIOS:
    cfg = load_config(sc, quick_test=QUICK_TEST)
    enhancer_name = cfg.get("enhancer", {}).get("name", None)
    has_enhancer = enhancer_name and enhancer_name.lower() != "none"

    run_dir = os.path.join(OUTPUT_ROOT, "runs", sn)
    weights_path = get_best_weights(run_dir)

    flops_dir = os.path.join(OUTPUT_ROOT, "evaluation", sn, "flops")

    # Load enhancer model if needed
    enhancer_model = None
    if has_enhancer:
        cache_dir = os.path.join(OUTPUT_ROOT, "model_cache")
        enhancer_obj = get_enhancer(enhancer_name, cache_dir)
        enhancer_obj.load_model()
        enhancer_model = enhancer_obj.model

    print(f"\n--- {sn} ---")
    raw = compute_all_flops(
        yolo_weights=weights_path,
        output_dir=flops_dir,
        scenario_name=sn,
        enhancer_model=enhancer_model,
        enhancer_name=enhancer_name if has_enhancer else None,
    )

    # Flatten for downstream (visualization, final table)
    flops_results[sn] = {
        "enhancer_gflops": raw.get("enhancer", {}).get("gflops", 0) or 0,
        "yolo_gflops": raw.get("yolo", {}).get("gflops", 0) or 0,
        "total_gflops": raw.get("total_gflops", 0) or 0,
    }

# Summary
flops_rows = []
for sn, res in flops_results.items():
    flops_rows.append({
        "Scenario": sn,
        "Enhancer GFLOPs": f"{res.get('enhancer_gflops', 0):.2f}",
        "YOLO GFLOPs": f"{res.get('yolo_gflops', 0):.2f}",
        "Total GFLOPs": f"{res.get('total_gflops', 0):.2f}",
    })
pd.DataFrame(flops_rows)


---
## Fase 7: Aggregation, Correlation & Visualization
Combine all results, compute Spearman correlations, generate publication figures.

In [ ]:
#@title Fase 7a: Spearman Correlation Analysis
from src.evaluation.correlation import compute_spearman_correlation

active_names = [sn for _, sn in ACTIVE_SCENARIOS]
corr_results = {}

if len(active_names) >= 3:
    corr_dir = os.path.join(OUTPUT_ROOT, "summary", "correlation")
    corr_results = compute_spearman_correlation(
        detection_results=eval_results,
        nr_results=nr_results,
        output_dir=corr_dir,
    )

    if "correlations" in corr_results:
        corr_rows = []
        for entry in corr_results["correlations"]:
            corr_rows.append({
                "NR Metric": entry["nr_metric"],
                "Det Metric": entry["det_metric"],
                "Spearman ρ": f"{entry['spearman_rho']:.4f}" if entry['spearman_rho'] else "N/A",
                "p-value": f"{entry['p_value']:.4f}" if entry['p_value'] else "N/A",
                "Interpretation": entry["interpretation"],
            })
        display(pd.DataFrame(corr_rows))
else:
    print(f"⚠ Correlation requires ≥3 scenarios. Currently active: {len(active_names)}")
    print("  → Uncomment more scenarios in ACTIVE_SCENARIOS (cell 0.1) and re-run all phases.")

In [ ]:
#@title Fase 7b: Generate All Figures
from src.utils.visualization import (
    plot_detection_comparison,
    plot_nr_metrics,
    plot_latency_breakdown,
    plot_correlation_scatter,
    export_detection_latex,
)

figures_dir = os.path.join(OUTPUT_ROOT, "summary", "figures")
os.makedirs(figures_dir, exist_ok=True)

active_names = [sn for _, sn in ACTIVE_SCENARIOS]

# Detection comparison bar chart
if eval_results:
    plot_detection_comparison(
        eval_results,
        os.path.join(figures_dir, "detection_comparison.png"),
    )

# NR metrics comparison
if nr_results:
    plot_nr_metrics(
        nr_results,
        os.path.join(figures_dir, "nr_metrics.png"),
    )

# Latency breakdown
if latency_results:
    plot_latency_breakdown(
        latency_results,
        os.path.join(figures_dir, "latency_breakdown.png"),
    )

# Correlation scatter plots (only if ≥3 scenarios)
if corr_results and "correlations" in corr_results:
    merged = {}
    for sn in active_names:
        if sn in eval_results and sn in nr_results:
            merged[sn] = {**eval_results[sn], **nr_results[sn]}

    for entry in corr_results["correlations"]:
        plot_correlation_scatter(
            data=merged,
            nr_metric=entry["nr_metric"],
            det_metric=entry["det_metric"],
            output_path=os.path.join(figures_dir, f"corr_{entry['nr_metric']}_vs_{entry['det_metric']}.png"),
            rho=entry.get("spearman_rho"),
            p_value=entry.get("p_value"),
        )

# LaTeX table
if eval_results:
    export_detection_latex(
        eval_results,
        os.path.join(OUTPUT_ROOT, "summary", "table_detection.tex"),
    )

print(f"\nAll figures saved in: {figures_dir}")

In [ ]:
#@title Fase 7c: Save Complete Summary
import json

summary_dir = os.path.join(OUTPUT_ROOT, "summary")
active_names = [sn for _, sn in ACTIVE_SCENARIOS]

# Compile master summary
master_summary = {
    "quick_test": QUICK_TEST,
    "active_scenarios": active_names,
    "scenarios": {},
}

for sn in active_names:
    master_summary["scenarios"][sn] = {
        "detection": eval_results.get(sn, {}),
        "nr_metrics": nr_results.get(sn, {}),
        "latency": latency_results.get(sn, {}),
        "flops": flops_results.get(sn, {}),
    }

summary_path = os.path.join(summary_dir, "master_summary.json")
os.makedirs(summary_dir, exist_ok=True)
with open(summary_path, "w") as f:
    json.dump(master_summary, f, indent=2, default=str)

print(f"Master summary saved: {summary_path}")
print(f"\n{'='*60}")
print(f"EXPERIMENTS COMPLETE for: {active_names}")
if len(active_names) < 4:
    print(f"\n→ Next: uncomment more scenarios in ACTIVE_SCENARIOS (cell 0.1)")
    print(f"  Then run Fase 2 (enhancement) + Fase 3 (training) for new scenarios")
    print(f"  Then re-run Fase 4+ to evaluate everything together")
print(f"{'='*60}")

---
## Final Summary Table

In [ ]:
#@title Display Final Comparison Table
active_names = [sn for _, sn in ACTIVE_SCENARIOS]

final_rows = []
for sn in active_names:
    det = eval_results.get(sn, {})
    nr = nr_results.get(sn, {})
    lat = latency_results.get(sn, {})
    fl = flops_results.get(sn, {})

    final_rows.append({
        "Scenario": sn,
        "mAP@0.5": f"{det.get('mAP_50', 0):.4f}",
        "mAP@0.5:0.95": f"{det.get('mAP_50_95', 0):.4f}",
        "NIQE ↓": f"{nr.get('niqe_mean', 0):.3f}",
        "BRISQUE ↓": f"{nr.get('brisque_mean', 0):.3f}",
        "T_total (ms)": f"{lat.get('T_total_mean', 0):.2f}",
        "GFLOPs": f"{fl.get('total_gflops', 0):.2f}",
    })

df_final = pd.DataFrame(final_rows)
print("\n" + "="*80)
print("FINAL COMPARISON TABLE")
print("="*80)
df_final